<a href="https://colab.research.google.com/github/shruthilakshmi008-BA/ba-automation-suite/blob/main/02.%20Requirement-Extractor/Requirement_Extractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai pandas

In [ ]:
import os
import re
import json
import pandas as pd

from google import genai

In [ ]:
from google.colab import files

uploaded = files.upload()

input_file = list(uploaded.keys())[0]

print(f"Input file uploaded: {input_file}")

Saving stakeholder_notes.txt to stakeholder_notes.txt
Input file uploaded: stakeholder_notes.txt


In [ ]:
with open(input_file, "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Stakeholder Notes:")
print("-" * 50)
print(raw_text)

Stakeholder Notes:
--------------------------------------------------
So basically the biggest issue right now is that when a purchase request
comes in from a branch, nobody knows who's supposed to approve it if
it's over 50k. Also people keep submitting requests without attaching
the vendor quote, and then finance has to chase them down which wastes
like two days every time. Oh and one more thing - the system doesn't
send any reminder if an approval sits pending for too long, so stuff
just sits there. We really need approvals to happen within 3 business
days max. Also would be nice if finance could see a simple dashboard of
all pending requests instead of digging through emails.



In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if GEMINI_API_KEY:
    print("Gemini API key loaded successfully.")
else:
    print("Gemini API key not found.")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client initialized successfully!")


Gemini client initialized successfully.


In [ ]:
def extract_requirements_gemini(raw_text):

    prompt = """
You are an expert Technical Business Analyst.

Analyze the following raw stakeholder discovery notes and extract
distinct, actionable requirements.

Return ONLY a valid JSON array.

Do not include:
- Markdown
- ```json
- Explanations
- Comments outside the JSON

Each requirement must contain exactly these fields:

1. requirement_id
   Example: REQ-001

2. description
   A clean, formal requirement statement.
   Remove filler phrases, conversational language, and transcript artifacts.

3. type
   Must be either:
   - Functional
   - Non-Functional

4. priority
   Must be either:
   - High
   - Medium
   - Low

Example:

[
    {
        "requirement_id": "REQ-001",
        "description": "The system should send reminders for pending requests.",
        "type": "Functional",
        "priority": "High"
    },
    {
        "requirement_id": "REQ-002",
        "description": "The system should respond to requests within two business days.",
        "type": "Non-Functional",
        "priority": "Medium"
    }
]

Stakeholder Notes:
"""

    try:
        response = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=prompt + "\n" + raw_text
        )

        result = response.text.strip()

        # Remove markdown code fences if Gemini adds them
        result = re.sub(
            r"^```json\s*|\s*```$",
            "",
            result,
            flags=re.IGNORECASE
        ).strip()

        requirements = json.loads(result)

        if not isinstance(requirements, list):
            raise ValueError("Gemini did not return a JSON list.")

        # Add extraction method
        for item in requirements:
            item["source_extraction_method"] = "Google Gemini"

        return requirements

    except Exception as e:
        print("Gemini extraction failed:")
        print(e)
        return None

In [ ]:
def split_into_candidates(raw_text):
    text = raw_text.replace("\n", " ").strip()

    raw_sentences = re.split(
        r"(?<=[.!?])\s+",
        text
    )

    return [
        s.strip()
        for s in raw_sentences
        if len(s.strip()) > 15
    ]


def infer_priority(sentence):
    lowered = sentence.lower()

    if any(k in lowered for k in [
        "biggest issue",
        "need",
        "max",
        "nobody knows"
    ]):
        return "High"

    if any(k in lowered for k in [
        "would be nice",
        "also would"
    ]):
        return "Low"

    return "Medium"


def infer_type(sentence):
    lowered = sentence.lower()

    if any(k in lowered for k in [
        "within",
        "days",
        "reminder",
        "sits pending"
    ]):
        return "Non-Functional"

    return "Functional"


def clean_description(sentence):
    cleaned = sentence.strip()

    fillers = [
        "So basically ",
        "Also ",
        "Oh and one more thing - ",
        "Also would be nice if ",
        "We really "
    ]

    for filler in fillers:
        if cleaned.startswith(filler):
            cleaned = cleaned[len(filler):]

    if cleaned:
        cleaned = cleaned[0].upper() + cleaned[1:]

    return cleaned

In [ ]:
def mock_extract(raw_text):

    candidates = split_into_candidates(raw_text)

    requirements = []

    for i, sentence in enumerate(candidates, start=1):

        requirements.append({
            "requirement_id": f"REQ-{i:03d}",
            "description": clean_description(sentence),
            "type": infer_type(sentence),
            "priority": infer_priority(sentence),
            "source_extraction_method": "Rule Engine (Fallback)"
        })

    return requirements

In [ ]:
requirements = extract_requirements_gemini(raw_text)

if requirements is None:
    print("\nUsing fallback rule-based extraction...")
    requirements = mock_extract(raw_text)

print(f"\nSuccessfully extracted {len(requirements)} requirements.")


Successfully extracted 5 requirements.


In [ ]:
df = pd.DataFrame(requirements)

df
output_file = "requirements_output.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print(f"Output saved successfully: {output_file}")

from google.colab import files

files.download(output_file)

Output saved successfully: requirements_output.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>